# Phase 2 — Node feature engineering
Loads `firm_years.parquet`, computes accounting / market / macro features per firm-year via the `node_processing/` modules, applies EBIT fallback (`ebit → oiadp → pi+xint`) and SIC-based Altman Z routing, then writes `node_features_raw.parquet` and runs validation.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
sys.path.append(str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from node_processing import (
    levereage_solvency as lev,
    profitability as prof,
    liquidity as liq,
    activity_efficiency as act,
    growth as grow,
    cash_flow as cf,
    composite_scores as cs,
)

## 1. Load `firm_years.parquet`
Only the columns needed for Phase 2 to keep memory in check.

In [ ]:
KEEP = [
    'gvkey', 'datadate', 'conm', 'sic', 'node_type',
    'at', 'ceq', 'sale', 'lct', 'lt', 'xint', 'rect', 'invt', 'cogs',
    'act', 'che', 'wcap', 'dlc', 'dltt',
    'ni', 'ebitda', 'gp', 'oiadp', 'pi', 'ebit', 're',
    'oancf', 'capx', 'csho', 'prcc_f', 'mkvalt', 'emp',
]
df = pd.read_parquet('firm_years.parquet', columns=KEEP)
df['datadate'] = pd.to_datetime(df['datadate'])
df = df.sort_values(['gvkey', 'datadate']).reset_index(drop=True)
print(f'{len(df):,} firm-years across {df["gvkey"].nunique():,} gvkeys')

## 2. Denominator hygiene
Replace zeros with NaN in fields that appear as denominators downstream (`at, ceq, sale, lct, lt, xint, rect, invt, cogs`). Once zeros are NaN, division produces NaN naturally — no need for the per-row `if denom == 0` guards inside the modules.

In [ ]:
ZERO_TO_NAN = ['at', 'ceq', 'sale', 'lct', 'lt', 'xint', 'rect', 'invt', 'cogs']
for c in ZERO_TO_NAN:
    df[c] = df[c].replace(0, np.nan)
df[ZERO_TO_NAN].isna().sum().to_frame('null_after_zero_replace')

## 3. Compute features module-by-module
Each module's scalar functions vectorize naturally over Series for plain arithmetic. A few cases with scalar `if`-guards or buggy comparisons are inlined with the same formula.

In [ ]:
feat = df[['gvkey', 'datadate', 'conm', 'sic', 'node_type']].copy()

### 3.1 Leverage / solvency

In [ ]:
# total_debt: lev.total_debt has a scalar zero-guard, so inline it
total_debt = df['dltt'].add(df['dlc'], fill_value=0)
total_debt = total_debt.where(total_debt != 0, np.nan)

feat['total_debt']        = total_debt
feat['debt_to_assets']    = lev.debt_to_assets(total_debt, df['at'])
feat['debt_to_equity']    = lev.debt_to_equity(total_debt, df['ceq'])
feat['lt_debt_ratio']     = df['dltt'] / df['at']                         # module fn returns a tuple — bug; use formula
feat['st_debt_ratio']     = lev.st_debt_ration(df['dlc'], df['at'])
feat['interest_coverage'] = lev.interest_coverage(df['ebitda'], df['xint'])
feat['st_debt_share']     = lev.st_debt_share(df['dlc'], total_debt)

### 3.2 Profitability

In [ ]:
feat['roa']                = prof.roa(df['ni'], df['at'])
feat['roe']                = prof.roe(df['ni'], df['ceq'])
feat['ebitda_margin']      = prof.ebitda_margin(df['ebitda'], df['sale'])
feat['gross_margin']       = prof.gross_margin(df['gp'], df['sale'])
feat['operational_margin'] = prof.operational_margin(df['oiadp'], df['sale'])

### 3.3 Liquidity
`liq.wc_to_assets` compares to `np.nan` with `==` (always False), so it never falls back to `act - lct`. Inline the intended logic.

In [ ]:
wc = df['wcap'].where(df['wcap'].notna(), df['act'] - df['lct'])

feat['current_ratio']  = liq.current_ratio(df['act'], df['lct'])
feat['quick_ratio']    = liq.quick_ration(df['che'], df['rect'], df['lct'])
feat['cash_to_assets'] = liq.cash_to_assets(df['che'], df['at'])
feat['wc_to_assets']   = wc / df['at']

### 3.4 Size
`np.log` of zero/negative values is `-inf` / `nan`. Replace nonpositives with NaN first.

In [ ]:
def safe_log(s):
    return np.log(s.where(s > 0))

feat['log_assets']  = safe_log(df['at'])
feat['log_revenue'] = safe_log(df['sale'])
feat['log_mktcap']  = safe_log(df['csho'] * df['prcc_f'])
feat['emp']         = df['emp']

### 3.5 Activity / efficiency

In [ ]:
feat['asset_turnover']       = act.asset_turnover(df['sale'], df['at'])
feat['receivables_turnover'] = act.receivables_turnover(df['sale'], df['rect'])
feat['inventory_turnover']   = act.inventory_turnover(df['cogs'], df['invt'])

### 3.6 Growth
Lag within `gvkey` after sorting by `datadate`. Treats consecutive rows as consecutive fiscal years.

In [ ]:
g = df.groupby('gvkey', sort=False)
sale_lag = g['sale'].shift(1)
at_lag   = g['at'].shift(1)
emp_lag  = g['emp'].shift(1)

feat['revenue_growth'] = grow.revenue_growth(df['sale'], sale_lag)
feat['asset_growth']   = grow.asset_growth(df['at'],   at_lag)
feat['emp_growth']     = grow.emp_growth(df['emp'],   emp_lag)

### 3.7 Cash flow

In [ ]:
feat['opcf_to_assets']  = cf.opcf_to_assets(df['oancf'], df['at'])
feat['capex_to_assets'] = cf.capex_to_assets(df['capx'], df['at'])
feat['fcf_to_assets']   = cf.fcf_to_assets(df['oancf'], df['capx'], df['at'])

## 4. EBIT fallback: `ebit → oiadp → pi + xint`
Compustat `ebit` is missing for many early-period rows and some financials. Fall back to `oiadp` (operating income after depreciation) and finally to `pi + xint` (pretax income + interest expense, the textbook reconstruction).

In [ ]:
ebit_proxy = df['ebit'].where(df['ebit'].notna(), df['oiadp'])
ebit_proxy = ebit_proxy.where(ebit_proxy.notna(), df['pi'].add(df['xint'], fill_value=np.nan))
feat['ebit_proxy'] = ebit_proxy

n = len(ebit_proxy)
src_ebit  = df['ebit'].notna().sum()
src_oiadp = (df['ebit'].isna() & df['oiadp'].notna()).sum()
src_pix   = (df['ebit'].isna() & df['oiadp'].isna() & ebit_proxy.notna()).sum()
print(f'ebit_proxy coverage: {ebit_proxy.notna().sum():,} / {n:,} ({ebit_proxy.notna().mean():.1%})')
print(f'  from ebit:    {src_ebit:>7,}')
print(f'  from oiadp:   {src_oiadp:>7,}')
print(f'  from pi+xint: {src_pix:>7,}')

## 5. Altman Z routing
Three Altman variants live in `composite_scores.py`. Routing rules:

| Group | SIC | Z variant | Equity input |
|---|---|---|---|
| Financial | 6000–6999 | NaN (excluded) | — |
| Manufacturer w/ market data | 2000–3999, has `csho`+`prcc_f` | `altman_z` (1968) | market value |
| Manufacturer w/o market data | 2000–3999, missing market data | `altman_z_prime` (1983) | book equity (`ceq`) |
| Other nonfinancial | else | `altman_z_double_prime` (1995) | book equity (`ceq`) |

The scalar Altman fns are vectorized via `np.vectorize`. Three masks split the universe.

In [ ]:
v_z   = np.vectorize(cs.altman_z,              otypes=[float])
v_zp  = np.vectorize(cs.altman_z_prime,        otypes=[float])
v_zpp = np.vectorize(cs.altman_z_double_prime, otypes=[float])

sic        = df['sic']
is_fin     = sic.between(6000, 6999)
is_mfr     = sic.between(2000, 3999)
has_market = df['csho'].notna() & df['prcc_f'].notna()

mfr_market_mask = (is_mfr & has_market & ~is_fin)
mfr_book_mask   = (is_mfr & ~has_market & ~is_fin)
other_nfn_mask  = (~is_mfr & ~is_fin)

altman_z_score = np.full(len(df), np.nan)
altman_variant = np.full(len(df), '', dtype=object)

# Manufacturers with market equity -> classic Z
m = mfr_market_mask.values
if m.any():
    altman_z_score[m] = v_z(
        df.loc[m, 'act'].values, df.loc[m, 'lct'].values,
        df.loc[m, 're'].values,  ebit_proxy[m].values,
        df.loc[m, 'csho'].values, df.loc[m, 'prcc_f'].values,
        df.loc[m, 'lt'].values,  df.loc[m, 'sale'].values,
        df.loc[m, 'at'].values,  df.loc[m, 'wcap'].values,
    )
    altman_variant[m] = 'z'

# Manufacturers without market equity -> Z' with book equity
m = mfr_book_mask.values
if m.any():
    altman_z_score[m] = v_zp(
        df.loc[m, 'act'].values, df.loc[m, 'lct'].values,
        df.loc[m, 're'].values,  ebit_proxy[m].values,
        df.loc[m, 'ceq'].values, df.loc[m, 'lt'].values,
        df.loc[m, 'sale'].values, df.loc[m, 'at'].values,
        df.loc[m, 'wcap'].values,
    )
    altman_variant[m] = 'z_prime'

# Other nonfinancials -> Z''
m = other_nfn_mask.values
if m.any():
    altman_z_score[m] = v_zpp(
        df.loc[m, 'act'].values, df.loc[m, 'lct'].values,
        df.loc[m, 're'].values,  ebit_proxy[m].values,
        df.loc[m, 'ceq'].values, df.loc[m, 'lt'].values,
        df.loc[m, 'at'].values,  df.loc[m, 'wcap'].values,
    )
    altman_variant[m] = 'z_double_prime'

# Financials -> stay nan, mark variant
altman_variant[is_fin.values] = 'financial_excluded'

feat['altman_z']       = altman_z_score
feat['altman_variant'] = altman_variant

# Zone label per variant
zone = np.full(len(df), '', dtype=object)
for var in ('z', 'z_prime', 'z_double_prime'):
    sel = (altman_variant == var)
    if sel.any():
        zone[sel] = [cs.z_zone(v, var) if not np.isnan(v) else np.nan
                     for v in altman_z_score[sel]]
feat['altman_zone'] = zone

print('Variant breakdown:')
print(feat['altman_variant'].value_counts(dropna=False).to_string())

## 6. S&P 500 trailing 12-month return
Attach **`sp500_ret_12m`** to each firm-year as a macro control. Source is `data/raw/sp500/sp500_monthly.csv` (yfinance ^GSPC daily resampled to month-start, 1980-01-01 → 2024-12-01). The trailing 12-month return is `close[t] / close[t-12] - 1`, then we as-of merge backward against `datadate`. Pre-1981 firm-years get NaN (no 12-month look-back possible).

In [ ]:
sp500_before = int(feat['sp500_ret_12m'].notna().sum()) if 'sp500_ret_12m' in feat.columns else 0
print(f'sp500_ret_12m before: {sp500_before:,} non-null '
      f'(annual notebook had no S&P 500 column previously)')

sp500 = pd.read_csv(PROJECT_ROOT / 'data/raw/sp500/sp500_monthly.csv')
sp500['date'] = pd.to_datetime(sp500['date'])
sp500 = sp500.sort_values('date').reset_index(drop=True)
sp500['sp500_ret_12m'] = sp500['close'].pct_change(periods=12)
sp500 = sp500[['date', 'sp500_ret_12m']]

# tolerance=45d so firm-years past the SP500 series end get NaN rather than stale carry-forward.
feat = feat.sort_values('datadate').reset_index(drop=True)
feat = pd.merge_asof(feat, sp500, left_on='datadate', right_on='date',
                     direction='backward', tolerance=pd.Timedelta(days=45))
feat = feat.drop(columns=['date'])

sp500_after = int(feat['sp500_ret_12m'].notna().sum())
print(f'sp500_ret_12m after:  {sp500_after:,} non-null '
      f'({sp500_after/len(feat):.1%})')

## 7. Replace infinities with NaN
`emp_growth` produces `inf` when prior-year `emp == 0`. Sweep all numeric columns to be safe.

In [ ]:
num_cols = feat.select_dtypes(include=[np.number]).columns
inf_before = {c: int(np.isinf(feat[c]).sum()) for c in num_cols if np.isinf(feat[c]).any()}
print('inf counts before cleanup:', inf_before)

feat[num_cols] = feat[num_cols].replace([np.inf, -np.inf], np.nan)

inf_after = {c: int(np.isinf(feat[c]).sum()) for c in num_cols if np.isinf(feat[c]).any()}
print('inf counts after cleanup: ', inf_after if inf_after else '(none)')

## 8. Save `node_features_raw.parquet`

In [ ]:
out_path = Path('node_features_raw.parquet')
feat.to_parquet(out_path, index=False)
print(f'wrote {out_path} — {feat.shape[0]:,} rows × {feat.shape[1]} cols, '
      f'{out_path.stat().st_size/1e6:.1f} MB')

## 9. Validation

### 9.1 Non-null counts per feature

In [ ]:
nn = feat.notna().sum().sort_values(ascending=False)
coverage = (nn / len(feat) * 100).round(1)
pd.DataFrame({'non_null': nn, 'coverage_pct': coverage})

### 9.2 Infinity check (post-cleanup, expect zero everywhere)

In [ ]:
num_cols = feat.select_dtypes(include=[np.number]).columns
{c: int(np.isinf(feat[c]).sum()) for c in num_cols if np.isinf(feat[c]).any()} or 'no infinities'

### 9.3 Spot-check Enron 2000 (gvkey 6127)
Enron filed Chapter 11 on 2 Dec 2001. The fiscal-2000 score should land in the **distress** zone if the pipeline is wired up correctly. SIC 5172 (wholesale petroleum) → routes to `altman_z_double_prime`.

In [ ]:
enron_2000 = feat[(feat['gvkey'] == 6127) & (feat['datadate'].dt.year == 2000)]
enron_2000.T

### 9.4 Sanity: a few defaulters from the Phase 1 known-defaulter list

In [ ]:
# Years are the last fiscal year before each filing.
# Note: Lehman Brothers is a financial firm (SIC 62xx) and is correctly excluded —
# substitute a nonfinancial defaulter (Kodak, filed Jan 2012).
targets = [
    ('ENRON',          2000),  # filed Dec 2001
    ('WORLDCOM',       2001),  # filed Jul 2002
    ('GENERAL MOTORS', 2008),  # filed Jun 2009
    ('EASTMAN KODAK',  2010),  # filed Jan 2012
]

rows = []
for kw, yr in targets:
    sub = feat[feat['conm'].str.contains(kw, regex=False, na=False) &
               (feat['datadate'].dt.year == yr)]
    if not sub.empty:
        rows.append(sub.iloc[[0]])
spot = pd.concat(rows)
spot[['conm', 'datadate', 'sic', 'altman_variant',
      'altman_z', 'altman_zone',
      'debt_to_assets', 'roa', 'log_assets']]

In [ ]:
# For any defaulter whose Z came back NaN, show which Z-input fields were missing in the raw data.
nan_rows = spot[spot['altman_z'].isna()][['gvkey', 'datadate']]
if nan_rows.empty:
    print('all defaulters scored — no NaN diagnostics needed')
    out = None
else:
    z_inputs = ['act', 'lct', 're', 'ebit', 'oiadp', 'pi', 'xint', 'csho', 'prcc_f',
                'ceq', 'lt', 'sale', 'at', 'wcap']
    diag = df.merge(nan_rows, on=['gvkey', 'datadate'])
    print('NaN-Z input field availability (True = missing):')
    out = diag[['conm', 'datadate'] + z_inputs].set_index(['conm', 'datadate']).isna().T
out